In [5]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from skimage.feature import hog
from sklearn.preprocessing import LabelEncoder

### Dataset Path

In [11]:
DATASET_PATH = "../dataset"

classes = os.listdir(DATASET_PATH)

### HOG Features

In [12]:
def extract_hog_features(image):

    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8,8),
        cells_per_block=(2,2),
        transform_sqrt=True,
        block_norm="L2-Hys"
    )

    return features

### Color Histogram

In [13]:
def extract_color_histogram(image):

    hist = cv2.calcHist(
        [image],
        [0,1,2],
        None,
        [8,8,8],
        [0,256,0,256,0,256]
    )

    hist = cv2.normalize(hist, hist).flatten()

    return hist

### Build Dataset Features

In [14]:
features = []
labels = []

IMAGE_SIZE = (224,224)

for class_name in classes:

    class_path = os.path.join(DATASET_PATH, class_name)

    for image_name in os.listdir(class_path):

        image_path = os.path.join(class_path, image_name)

        image = cv2.imread(image_path)

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        image = cv2.resize(image, IMAGE_SIZE)

        # Extract features
        hog_features = extract_hog_features(image)

        color_features = extract_color_histogram(image)

        # Combine features
        combined_features = np.hstack([
            hog_features,
            color_features
        ])

        features.append(combined_features)

        labels.append(class_name)

### Convert Arrays

In [15]:
X = np.array(features)

y = np.array(labels)

print("Feature matrix shape :", X.shape)

print("Labels shape :", y.shape)

Feature matrix shape : (6652, 26756)
Labels shape : (6652,)


### Encode Labels

In [17]:
encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

### Save Features

In [18]:
np.save("../models/X_features.npy", X)

np.save("../models/y_labels.npy", y_encoded)